In [1]:
import pandas as pd
import numpy as np;
import glob;
import os;
from scipy.spatial import cKDTree
from collections import defaultdict

def merge_data(bushfire_df, weather_df: pd.DataFrame):
    print("Converting dates...")
    bushfire_df['discovery_date'] = pd.to_datetime(bushfire_df['discovery_date']).dt.date
    weather_df['date'] = pd.to_datetime(weather_df['date']).dt.date

    print("Grouping weather data...")
    weather_grouped = weather_df.groupby(['date', 'state'])

    print("Processing bushfires...")
    results = []
    total_fires = len(bushfire_df)
    missing_data_log = defaultdict(int)

    for i, (idx, fire) in enumerate(bushfire_df.iterrows()):
        if i % 1000 == 0:
            print(f"Processing fire {i+1}/{total_fires}")

        try:
            day_state_weather = weather_grouped.get_group((fire['discovery_date'], fire['state']))
            
            if not day_state_weather.empty:
                tree = cKDTree(day_state_weather[['latitude', 'longitude']])
                distance, index = tree.query([fire['latitude'], fire['longitude']])
                nearest_station = day_state_weather.iloc[index]
                
                results.append({
                    **fire.to_dict(),
                    't_min': nearest_station['t_min'],
                    't_max': nearest_station['t_max'],
                    'elevation': nearest_station['elevation']
                })
            else:
                missing_data_log[(fire['discovery_date'], fire['state'])] += 1
                results.append(fire.to_dict())
        except KeyError:
            missing_data_log[(fire['discovery_date'], fire['state'])] += 1
            results.append(fire.to_dict())

    print("Creating merged DataFrame...")
    merged_df = pd.DataFrame(results)

    print("\nMissing Weather Data Summary:")
    for (date, state), count in missing_data_log.items():
        print(f"Date: {date}, State: {state}, Missing Entries: {count}")

    return merged_df


In [2]:

# Let's load our weather and bushfire data:
weather_files = glob.glob('./../output-final/*_weather_data.csv')
bushfire_df = pd.read_csv('./cleaned-bushfire-data.csv')


In [3]:
merged_dfs = []

# Loop through each weather file
for weather_file in weather_files:
    year = os.path.basename(weather_file).split('_')[0]  # Extract year from filename
    print(f"Processing weather data for year {year}")
    
    # Read weather data for the current year
    weather_df = pd.read_csv(weather_file)
    
    # Merge bushfire data with weather data for this year
    merged_df = merge_data(bushfire_df, weather_df)
    
    # Append the result to our list
    merged_dfs.append(merged_df)

# Concatenate all merged dataframes
final_merged_df = pd.concat(merged_dfs, ignore_index=True)

# Remove any potential duplicates
final_merged_df = final_merged_df.drop_duplicates()

# Let's save the final merged dataset
final_merged_df.to_csv('~/Desktop/merged_bushfire_weather_data.csv', index=False)

Processing weather data for year 2002
Converting dates...
Grouping weather data...
Processing bushfires...
Processing fire 1/235820
Processing fire 1001/235820
Processing fire 2001/235820
Processing fire 3001/235820
Processing fire 4001/235820
Processing fire 5001/235820
Processing fire 6001/235820
Processing fire 7001/235820
Processing fire 8001/235820
Processing fire 9001/235820
Processing fire 10001/235820
Processing fire 11001/235820
Processing fire 12001/235820
Processing fire 13001/235820
Processing fire 14001/235820
Processing fire 15001/235820
Processing fire 16001/235820
Processing fire 17001/235820
Processing fire 18001/235820
Processing fire 19001/235820
Processing fire 20001/235820
Processing fire 21001/235820
Processing fire 22001/235820
Processing fire 23001/235820
Processing fire 24001/235820
Processing fire 25001/235820
Processing fire 26001/235820
Processing fire 27001/235820
Processing fire 28001/235820
Processing fire 29001/235820
Processing fire 30001/235820
Process

KeyboardInterrupt: 

In [ ]:
def impute_elevation(df: pd.DataFrame):
    # Calculate the median elevation for each state
    state_median_elevation = df.groupby('state')['elevation'].median()

    # For each state
    for state in df['state'].unique():
        state_data = df[df['state'] == state]
        
        # If all elevations are NaN for this state, use the global median
        if state_data['elevation'].isna().all():
            fill_value = df['elevation'].median()
        else:
            fill_value = state_median_elevation[state]
        
        # Fill NaN values for this state
        df.loc[(df['state'] == state) & (df['elevation'].isna()), 'elevation'] = fill_value

    # If there are still NaN values, fill with global median
    df['elevation'].fillna(df['elevation'].median(), inplace=True)

    return df

def impute_temperature(df: pd.DataFrame, column):
    # Calculate median temperatures for each state and month
    state_month_median = df.groupby(['state', 'discovery_month'])[column].median()

    # For each state and month combination
    for state in df['state'].unique():
        for month in df['discovery_month'].unique():
            mask = (df['state'] == state) & (df['discovery_month'] == month)
            
            # If all temperatures are NaN for this state and month, use the state median
            if df.loc[mask, column].isna().all():
                fill_value = df[df['state'] == state][column].median()
                if pd.isna(fill_value):  # If state median is also NaN, use global median
                    fill_value = df[column].median()
            else:
                fill_value = state_month_median.loc[state, month]
            
            # Fill NaN values for this state and month
            df.loc[mask & df[column].isna(), column] = fill_value

    # If there are still NaN values, fill with global median
    df[column].fillna(df[column].median(), inplace=True)

    return df

# Apply the imputation
print("Imputing elevation data...")
final_merged_df = impute_elevation(final_merged_df)

print("Imputing temperature data...")
final_merged_df = impute_temperature(final_merged_df, 't_min')
final_merged_df = impute_temperature(final_merged_df, 't_max')

# Verify that all NaN values have been handled
print("\nChecking for remaining NaN values:")
print(final_merged_df[['elevation', 't_min', 't_max']].isnull().sum())

# Ensure t_max is always greater than or equal to t_min
print("\nEnsuring t_max >= t_min...")
final_merged_df.loc[final_merged_df['t_max'] < final_merged_df['t_min'], 't_max'] = final_merged_df['t_min']

print("\nData imputation complete.")

In [8]:
# # Here's our final merged data =)
# newly_merged_cleaned = final_merged_df.dropna(subset=['t_min', 't_max', 'elevation'])

In [9]:
# def impute_temperature(df, column):
#     # First, try to fill with the mean of the same day and state
#     df[column] = df.groupby(['state', 'discovery_doy'])[column].transform(lambda x: x.fillna(x.mean()))
    
#     # If still NaN, use the mean of the same month and state
#     df[column] = df.groupby(['state', 'discovery_month'])[column].transform(lambda x: x.fillna(x.mean()))
    
#     # If still NaN, use the overall state mean
#     df[column] = df.groupby('state')[column].transform(lambda x: x.fillna(x.mean()))
    
#     return df

# final_merged_df = impute_temperature(final_merged_df, 't_min')
# final_merged_df = impute_temperature(final_merged_df, 't_max')

In [10]:
# import numpy as np

# def impute_elevation(df):
#     # Calculate the median elevation for each state
#     state_median_elevation = df.groupby('state')['elevation'].median()

#     # For each state
#     for state in df['state'].unique():
#         state_data = df[df['state'] == state]
        
#         # If all elevations are NaN for this state, use the global median
#         if state_data['elevation'].isna().all():
#             fill_value = df['elevation'].median()
#         else:
#             fill_value = state_median_elevation[state]
        
#         # Fill NaN values for this state
#         df.loc[(df['state'] == state) & (df['elevation'].isna()), 'elevation'] = fill_value

#     # If there are still NaN values, fill with global median
#     df['elevation'].fillna(df['elevation'].median(), inplace=True)

#     return df

# # Apply the imputation
# final_merged_df = impute_elevation(final_merged_df)

# # Check if there are any remaining NaN values
# print(final_merged_df['elevation'].isna().sum())

In [11]:
# mask = newly_merged_cleaned['t_min'] > newly_merged_cleaned['t_max']
# newly_merged_cleaned.loc[mask, ['t_min', 't_max']] = newly_merged_cleaned.loc[mask, ['t_max', 't_min']].values

In [12]:
# newly_merged_cleaned.loc[newly_merged_cleaned['t_min'] > 10]

In [13]:
# newly_merged_cleaned.to_csv('merged_data.csv', index=False)

In [14]:
# df = pd.read_csv('./final_weather_data/2000_weather_data.csv')

# df = pd.read_csv('~/Desktop/weather-data/2000.csv')

# df.sample(5)
# df.loc[df['T_MIN'] > 60]